In [ ]:
# Cell 1: Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gradio', 'speechbrain', 'librosa', 'soundfile'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
print('All packages installed.')

In [ ]:
# Cell 2: Imports & config
import os, json, glob, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
import gradio as gr
from pathlib import Path
warnings.filterwarnings('ignore')

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_RATE = 16000
N_MFCC      = 40
MAX_LEN     = 300
print(f'Device: {DEVICE}')

# ── Auto-find Phase 5 outputs ─────────────────────────────────────────────────
def _find(name):
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) +
            glob.glob(f'/kaggle/working/**/{name}', recursive=True))
    return hits[0] if hits else None

PHASE2_CKPT  = _find('tdnn_best.pt')
PHASE4_CKPT  = _find('antispoof_best.pt')
CENTROID_P2  = _find('enrollment_centroid_p2.npy')
CENTROID_P3  = _find('enrollment_centroid_p3.npy')
SUMMARY_PATH = _find('phase5_summary.json')

print(f'Phase 2 ckpt  : {PHASE2_CKPT}')
print(f'Phase 4 ckpt  : {PHASE4_CKPT}')
print(f'Centroid P2   : {CENTROID_P2}')
print(f'Centroid P3   : {CENTROID_P3}')
print(f'Phase 5 summary: {SUMMARY_PATH}')

In [ ]:
# Cell 3: Rebuild model architectures (same as Phase 5)
import torch, torch.nn as nn, torch.nn.functional as F

# ── TDNN (Phase 2) ────────────────────────────────────────────────────────────
class _TDNNLayer(nn.Module):
    def __init__(self, in_d, out_d, ctx=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(in_d, out_d, kernel_size=ctx, dilation=dil,
                              padding=(ctx-1)*dil//2)
        self.bn = nn.BatchNorm1d(out_d)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))

class TDNNSpeakerNet(nn.Module):
    def __init__(self, in_d=40, emb_d=256):
        super().__init__()
        self.tdnn = nn.Sequential(
            _TDNNLayer(in_d, 512, 5, 1), _TDNNLayer(512, 512, 3, 2),
            _TDNNLayer(512, 512, 3, 3),  _TDNNLayer(512, 512, 1, 1),
            _TDNNLayer(512, 1500, 1, 1),
        )
        self.fc1 = nn.Sequential(nn.Linear(3000, 512), nn.BatchNorm1d(512),
                                  nn.ReLU(), nn.Dropout(0.2))
        self.fc2 = nn.Sequential(nn.Linear(512, emb_d), nn.BatchNorm1d(emb_d))
    def forward(self, x):
        x = self.tdnn(x.transpose(1, 2))
        x = torch.cat([x.mean(2), x.std(2)], 1)
        return F.normalize(self.fc2(self.fc1(x)), dim=1)

# ── AntiSpoofNet (Phase 4) ────────────────────────────────────────────────────
class FMS(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.scale = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                    nn.Linear(ch, ch), nn.Sigmoid())
    def forward(self, x): return x * self.scale(x).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.fms   = FMS(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        self.drop  = nn.Dropout(0.1)
    def forward(self, x):
        h = F.leaky_relu(self.bn1(self.conv1(x)), 0.1)
        h = self.drop(self.bn2(self.conv2(h)))
        return F.leaky_relu(self.fms(h) + self.skip(x), 0.1)

class AntiSpoofNet(nn.Module):
    def __init__(self, input_dim=120):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.LeakyReLU(0.1)
        )
        self.layer1 = ResBlock1D(64,  128, stride=2)
        self.layer2 = ResBlock1D(128, 128)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 256)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = self.stem(x)
        h = self.layer1(h); h = self.layer2(h)
        h = self.layer3(h); h = self.layer4(h)
        h = torch.cat([h.mean(-1), h.std(-1) + 1e-8], -1)
        return self.classifier(h).squeeze(-1)

print('Architectures defined.')

In [ ]:
# Cell 4: Load all models
import os, json, torch, torch.nn as nn, torch.nn.functional as F

# ── Architecture definitions (self-contained) ─────────────────────────────────
class _TDNNLayer(nn.Module):
    def __init__(self, in_d, out_d, ctx=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(in_d, out_d, kernel_size=ctx, dilation=dil,
                              padding=(ctx-1)*dil//2)
        self.bn = nn.BatchNorm1d(out_d)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))

class TDNNSpeakerNet(nn.Module):
    def __init__(self, in_d=40, emb_d=256):
        super().__init__()
        self.tdnn = nn.Sequential(
            _TDNNLayer(in_d, 512, 5, 1), _TDNNLayer(512, 512, 3, 2),
            _TDNNLayer(512, 512, 3, 3),  _TDNNLayer(512, 512, 1, 1),
            _TDNNLayer(512, 1500, 1, 1),
        )
        self.fc1 = nn.Sequential(nn.Linear(3000, 512), nn.BatchNorm1d(512),
                                  nn.ReLU(), nn.Dropout(0.2))
        self.fc2 = nn.Sequential(nn.Linear(512, emb_d), nn.BatchNorm1d(emb_d))
    def forward(self, x):
        x = self.tdnn(x.transpose(1, 2))
        x = torch.cat([x.mean(2), x.std(2)], 1)
        return F.normalize(self.fc2(self.fc1(x)), dim=1)

class FMS(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.scale = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                    nn.Linear(ch, ch), nn.Sigmoid())
    def forward(self, x): return x * self.scale(x).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.fms   = FMS(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        self.drop  = nn.Dropout(0.1)
    def forward(self, x):
        h = F.leaky_relu(self.bn1(self.conv1(x)), 0.1)
        h = self.drop(self.bn2(self.conv2(h)))
        return F.leaky_relu(self.fms(h) + self.skip(x), 0.1)

class AntiSpoofNet(nn.Module):
    def __init__(self, input_dim=120):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.LeakyReLU(0.1)
        )
        self.layer1 = ResBlock1D(64,  128, stride=2)
        self.layer2 = ResBlock1D(128, 128)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 256)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1), nn.Dropout(0.3), nn.Linear(256, 1)
        )
    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = self.stem(x)
        h = self.layer1(h); h = self.layer2(h)
        h = self.layer3(h); h = self.layer4(h)
        h = torch.cat([h.mean(-1), h.std(-1) + 1e-8], -1)
        return self.classifier(h).squeeze(-1)

# ── Phase 2: TDNN ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tdnn_model = None
if PHASE2_CKPT and os.path.exists(PHASE2_CKPT):
    state = torch.load(PHASE2_CKPT, map_location=DEVICE)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    tdnn_model = TDNNSpeakerNet().to(DEVICE)
    tdnn_model.load_state_dict(state)
    tdnn_model.eval()
    print(f'TDNN loaded: {PHASE2_CKPT}')
else:
    print('WARNING: TDNN checkpoint not found.')

# ── Phase 3: ECAPA-TDNN ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'speechbrain'], check=True)
from speechbrain.inference.speaker import SpeakerRecognition
ecapa_model = SpeakerRecognition.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='/kaggle/working/pretrained_ecapa'
)
print('ECAPA-TDNN loaded.')

# ── Phase 4: AntiSpoofNet ─────────────────────────────────────────────────────
spoof_model = None
if PHASE4_CKPT and os.path.exists(PHASE4_CKPT):
    state4 = torch.load(PHASE4_CKPT, map_location=DEVICE)
    if isinstance(state4, dict) and 'model_state_dict' in state4:
        state4 = state4['model_state_dict']
    spoof_model = AntiSpoofNet(input_dim=120).to(DEVICE)
    spoof_model.load_state_dict(state4)
    spoof_model.eval()
    print(f'AntiSpoofNet loaded: {PHASE4_CKPT}')
else:
    print('WARNING: AntiSpoofNet checkpoint not found.')

# ── Load fusion weights & threshold from Phase 5 ─────────────────────────────
W2, W3, W4 = 0.33, 0.34, 0.33
DEFAULT_THRESHOLD = 0.5

if SUMMARY_PATH and os.path.exists(SUMMARY_PATH):
    with open(SUMMARY_PATH) as f:
        p5 = json.load(f)
    W2 = p5['weights']['p2']
    W3 = p5['weights']['p3']
    W4 = p5['weights']['p4']
    DEFAULT_THRESHOLD = p5['optimal_threshold']
    print(f'Phase 5 weights loaded: P2={W2:.3f} P3={W3:.3f} P4={W4:.3f}')
    print(f'Optimal threshold: {DEFAULT_THRESHOLD:.4f}')
else:
    print('Phase 5 summary not found — using equal weights and threshold=0.5')

print('\nAll models ready.')

In [ ]:
# Cell 5: Feature extraction & scoring functions
import torchaudio

def load_audio(path):
    wav, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return wav

def mfcc_p2(wav):
    feat = librosa.feature.mfcc(y=wav, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                  n_fft=512, hop_length=160, win_length=400)
    feat = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    T = feat.shape[1]
    feat = feat[:, :MAX_LEN] if T >= MAX_LEN else np.pad(feat, ((0,0),(0,MAX_LEN-T)))
    return feat.T.astype(np.float32)  # (T, 40)

def mfcc_p4(wav):
    target = MAX_LEN * 160
    wav = np.pad(wav, (0, max(0, target - len(wav))))[:target]
    mfcc   = librosa.feature.mfcc(y=wav, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                    n_fft=512, hop_length=160, n_mels=80)
    delta  = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.concatenate([mfcc, delta, delta2], axis=0)
    feat   = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    T = feat.shape[1]
    feat = feat[:, :MAX_LEN] if T >= MAX_LEN else np.pad(feat, ((0,0),(0,MAX_LEN-T)))
    return feat.T.astype(np.float32)  # (T, 120)

@torch.no_grad()
def get_tdnn_emb(wav):
    if tdnn_model is None: return np.zeros(256)
    t = torch.FloatTensor(mfcc_p2(wav)).unsqueeze(0).to(DEVICE)
    return tdnn_model(t).cpu().numpy().squeeze()

@torch.no_grad()
def get_ecapa_emb(wav):
    t = torch.FloatTensor(wav).unsqueeze(0)
    emb = ecapa_model.encode_batch(t)
    return emb.squeeze().cpu().numpy()

@torch.no_grad()
def get_spoof_score(wav):
    if spoof_model is None: return 0.5
    t = torch.FloatTensor(mfcc_p4(wav)).unsqueeze(0).to(DEVICE)
    return float(torch.sigmoid(spoof_model(t)).cpu().item())

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def fuse(s2, s3, s4):
    s2n = (s2 + 1) / 2
    s3n = (s3 + 1) / 2
    return float(W2 * s2n + W3 * s3n + W4 * s4)

print('Scoring functions ready.')

In [ ]:
# Cell 6: Enrollment state (in-memory)
enrollment_state = {
    'centroid_p2': None,
    'centroid_p3': None,
    'n_samples':   0,
    'embeddings_p2': [],
    'embeddings_p3': [],
}

# Pre-load saved centroids from Phase 5 if available
if CENTROID_P2 and os.path.exists(CENTROID_P2):
    enrollment_state['centroid_p2'] = np.load(CENTROID_P2)
    print(f'Pre-loaded P2 centroid from Phase 5: {enrollment_state["centroid_p2"].shape}')
if CENTROID_P3 and os.path.exists(CENTROID_P3):
    enrollment_state['centroid_p3'] = np.load(CENTROID_P3)
    enrollment_state['n_samples']   = 1
    print(f'Pre-loaded P3 centroid from Phase 5: {enrollment_state["centroid_p3"].shape}')

def enroll(audio_path):
    wav = load_audio(audio_path)
    e2  = get_tdnn_emb(wav)
    e3  = get_ecapa_emb(wav)
    enrollment_state['embeddings_p2'].append(e2)
    enrollment_state['embeddings_p3'].append(e3)
    enrollment_state['centroid_p2'] = np.mean(enrollment_state['embeddings_p2'], axis=0)
    enrollment_state['centroid_p3'] = np.mean(enrollment_state['embeddings_p3'], axis=0)
    enrollment_state['n_samples']  += 1
    return enrollment_state['n_samples']

def verify(audio_path, threshold):
    if enrollment_state['centroid_p2'] is None or enrollment_state['centroid_p3'] is None:
        return None, None, None, None, False, 'Not enrolled yet'
    wav = load_audio(audio_path)
    s2  = cosine_sim(get_tdnn_emb(wav),  enrollment_state['centroid_p2'])
    s3  = cosine_sim(get_ecapa_emb(wav), enrollment_state['centroid_p3'])
    s4  = get_spoof_score(wav)
    fs  = fuse(s2, s3, s4)
    accepted = fs >= threshold
    return s2, s3, s4, fs, accepted, ''

print('Enrollment state ready.')

In [ ]:
# Cell 7: Gradio UI

CSS = """
.accept  { background: #d4edda !important; border: 2px solid #28a745 !important;
           color: #155724 !important; font-size: 1.4em !important; font-weight: bold !important; }
.reject  { background: #f8d7da !important; border: 2px solid #dc3545 !important;
           color: #721c24 !important; font-size: 1.4em !important; font-weight: bold !important; }
.spoof   { background: #fff3cd !important; border: 2px solid #ffc107 !important;
           color: #856404 !important; font-size: 1.4em !important; font-weight: bold !important; }
"""

def ui_enroll(audio):
    if audio is None:
        return '⚠️ No audio provided.', gr.update()
    n = enroll(audio)
    msg = f'✅ Enrolled sample {n}/3.  '
    if n >= 3:
        msg += '🔒 Speaker profile ready! You can now verify.'
    else:
        msg += f'Add {3 - n} more sample(s) for a stronger profile.'
    return msg, gr.update(value=f'Enrolled: {n} sample(s)')

def ui_reset():
    enrollment_state['centroid_p2']   = None
    enrollment_state['centroid_p3']   = None
    enrollment_state['n_samples']     = 0
    enrollment_state['embeddings_p2'] = []
    enrollment_state['embeddings_p3'] = []
    return '🗑️ Enrollment cleared.', gr.update(value='Not enrolled')

def ui_verify(audio, threshold):
    if audio is None:
        return '⚠️ No audio provided.', '', '', '', ''
    s2, s3, s4, fs, accepted, err = verify(audio, threshold)
    if err:
        return f'⚠️ {err}', '', '', '', ''

    p2_pct = (s2 + 1) / 2 * 100
    p3_pct = (s3 + 1) / 2 * 100
    p4_pct = s4 * 100
    fs_pct = fs * 100

    spoof_flag = s4 < 0.3

    if spoof_flag:
        verdict = '🚨 SPOOF DETECTED — REJECTED'
        css_cls = 'spoof'
    elif accepted:
        verdict = '✅ ACCESS GRANTED'
        css_cls = 'accept'
    else:
        verdict = '❌ ACCESS DENIED'
        css_cls = 'reject'

    scores_md = (
        f'| System | Score |\n'
        f'|--------|-------|\n'
        f'| Phase 2 — TDNN (text-dep) | {p2_pct:.1f}% |\n'
        f'| Phase 3 — ECAPA (text-indep) | {p3_pct:.1f}% |\n'
        f'| Phase 4 — Anti-Spoof | {p4_pct:.1f}% |\n'
        f'| **Fused Score** | **{fs_pct:.1f}%** |\n'
        f'| Threshold | {threshold*100:.1f}% |'
    )

    bars_md = (
        f'**TDNN**  {_make_bar(p2_pct)} {p2_pct:.0f}%\n\n'
        f'**ECAPA** {_make_bar(p3_pct)} {p3_pct:.0f}%\n\n'
        f'**Anti-Spoof** {_make_bar(p4_pct)} {p4_pct:.0f}%\n\n'
        f'**Fused** {_make_bar(fs_pct, bold=True)} {fs_pct:.0f}%'
    )

    return verdict, css_cls, scores_md, bars_md, f'Spoof flag: {spoof_flag}'

def _make_bar(pct, bold=False):
    filled = int(pct / 5)
    bar = '█' * filled + '░' * (20 - filled)
    return f'`{bar}`'

# ── Build Gradio interface ────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title='Voice Biometric Authentication') as demo:
    gr.Markdown('# 🎙️ Voice Biometric Authentication System')
    gr.Markdown('**Phase 2** (TDNN) + **Phase 3** (ECAPA-TDNN) + **Phase 4** (Anti-Spoofing) + **Phase 5** (Score Fusion)')

    with gr.Row():
        # ── Left panel: Enrollment ─────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('## 📝 Step 1: Enroll Your Voice')
            gr.Markdown('Record or upload 3 voice samples (any phrase each time).')

            enroll_audio  = gr.Audio(
                sources=['microphone', 'upload'],
                type='filepath',
                label='Record or upload enrollment sample'
            )
            enroll_btn    = gr.Button('➕ Add Enrollment Sample', variant='primary')
            reset_btn     = gr.Button('🗑️ Reset Enrollment', variant='stop')
            enroll_status = gr.Textbox(label='Status', value='Not enrolled', interactive=False)
            enroll_msg    = gr.Textbox(label='Message', interactive=False)

        # ── Right panel: Verification ──────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('## 🔍 Step 2: Verify')
            gr.Markdown('Record or upload your voice — the system will accept or reject it.')

            verify_audio = gr.Audio(
                sources=['microphone', 'upload'],
                type='filepath',
                label='Record or upload voice to verify'
            )
            threshold_sl = gr.Slider(minimum=0.3, maximum=0.9, value=DEFAULT_THRESHOLD,
                                      step=0.01, label=f'Threshold (default={DEFAULT_THRESHOLD:.2f})')
            verify_btn   = gr.Button('🔐 Verify', variant='primary')

            verdict_box  = gr.Textbox(label='Decision', interactive=False,
                                       elem_classes=['accept'])
            scores_box   = gr.Markdown(label='Score breakdown')
            bars_box     = gr.Markdown(label='Score bars')

    # ── Event handlers ─────────────────────────────────────────────────────────
    enroll_btn.click(
        fn=ui_enroll,
        inputs=[enroll_audio],
        outputs=[enroll_msg, enroll_status]
    )
    reset_btn.click(
        fn=ui_reset,
        inputs=[],
        outputs=[enroll_msg, enroll_status]
    )
    verify_btn.click(
        fn=ui_verify,
        inputs=[verify_audio, threshold_sl],
        outputs=[verdict_box, gr.Textbox(visible=False),
                  scores_box, bars_box, gr.Textbox(visible=False)]
    )

print('Gradio UI built. Launching...')
demo.launch(share=True, debug=False)